# TarkeebLM — Phase A: SFT on ResPlan (Kaggle, background-safe)

Same pipeline as the Colab notebook, adapted for Kaggle so it runs **with your computer OFF**:
use **Save Version → Save & Run All (Commit)** and Kaggle executes the whole notebook on its
servers; results land in the notebook's **Output** tab.

**Setup (right panel): Accelerator = GPU T4 x2 or P100 · Internet = ON** (needs phone-verified account).

Trains Qwen3-8B (fits Kaggle's 16 GB) with QLoRA + Unsloth on ResPlan converted to
TarkeebAI `plan_schema.json`. LoRA is zipped into `/kaggle/working` (downloadable later).

In [ ]:
# ── 1. GPU check ───────────────────────────────────────────────────────
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU: {gpu} | VRAM: {vram:.0f} GB')
MODEL = 'unsloth/Qwen3-14B-unsloth-bnb-4bit' if vram >= 20 else 'unsloth/Qwen3-8B-unsloth-bnb-4bit'
MAX_SEQ = 4096
print('MODEL =', MODEL)

In [ ]:
# ── 2. Install ─────────────────────────────────────────────────────────
%%capture
!pip install -q unsloth shapely jsonschema

In [ ]:
# ── 3. Data: clone + convert ───────────────────────────────────────────
import os
if not os.path.exists('ResPlan'):
    !git clone --depth 1 https://github.com/m-agour/ResPlan.git
if not os.path.exists('PlanForgeRevit'):
    !git clone --depth 1 https://github.com/mhmdTaqi-code/PlanForgeRevit.git
!cd ResPlan && unzip -n -q ResPlan.zip
PKL = !find ResPlan -name '*.pkl' -maxdepth 2
PKL = PKL[0]
print('pickle:', PKL)

!python PlanForgeRevit/tools/resplan_to_tarkeeb.py \
    --pickle "$PKL" \
    --schema PlanForgeRevit/schema/plan_schema.json \
    --out data/tarkeeblm --max-plans 20000

In [ ]:
# ── 4. Load model (QLoRA) ──────────────────────────────────────────────
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=17)

In [ ]:
# ── 5. Dataset -> chat format ──────────────────────────────────────────
import json
from datasets import Dataset

SYSTEM = ('You are TarkeebLM, an architectural floor-plan generator. '
          'Respond with ONLY a JSON object conforming to the TarkeebAI plan_schema '
          '(meta, levels, rooms with polygon coordinates in meters, walls, doors, windows). '
          'No prose, no markdown.')

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8')]

def to_text(row):
    msgs = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': row['prompt']},
        {'role': 'assistant', 'content': row['target']},
    ]
    return {'text': tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=False, enable_thinking=False)}

train_rows = load_jsonl('data/tarkeeblm/train.jsonl')
val_rows = load_jsonl('data/tarkeeblm/val.jsonl')
train_ds = Dataset.from_list(train_rows).map(to_text, remove_columns=['prompt','target'])
print(len(train_rows), 'train |', len(val_rows), 'val')

In [ ]:
# ── 6. Train ───────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds,
    dataset_text_field='text',
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type='cosine',
        warmup_ratio=0.03, logging_steps=25, save_steps=500,
        output_dir='ckpt', optim='adamw_8bit', seed=17,
        max_seq_length=MAX_SEQ, packing=False, report_to='none'))
trainer.train()

In [ ]:
# ── 7. Eval: validity / schema / program fidelity ──────────────────────
import json, re, jsonschema
FastLanguageModel.for_inference(model)
schema = json.load(open('PlanForgeRevit/schema/plan_schema.json'))
validator = jsonschema.Draft202012Validator(schema)

def generate(prompt):
    msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':prompt}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                        enable_thinking=False, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=ids, max_new_tokens=2500, temperature=0.7,
                         top_p=0.9, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

N, parsed, valid, fidelity = 30, 0, 0, 0
for row in val_rows[:N]:
    txt = generate(row['prompt'])
    m = re.search(r'\{.*\}', txt, re.S)
    if not m:
        continue
    try:
        plan = json.loads(m.group(0)); parsed += 1
    except Exception:
        continue
    if next(validator.iter_errors(plan), None) is None:
        valid += 1
        want = json.loads(row['target'])
        wc = sorted(r['type'] for r in want['rooms'])
        gc = sorted(r['type'] for r in plan.get('rooms', []))
        fidelity += int(wc == gc)
print(f'RESULTS >>> JSON parse: {parsed}/{N} | schema valid: {valid}/{N} | program match: {fidelity}/{N}')

In [ ]:
# ── 8. Save LoRA into the notebook Output (no Drive needed) ────────────
OUT = '/kaggle/working/TarkeebLM_v0'
model.save_pretrained(OUT); tokenizer.save_pretrained(OUT)
!cd /kaggle/working && zip -r -q TarkeebLM_v0.zip TarkeebLM_v0 && ls -lh TarkeebLM_v0.zip
print('LoRA zipped into the notebook Output tab — downloadable from kaggle.com later')

In [ ]:
# ── 9. Demo ────────────────────────────────────────────────────────────
demo = ('Design a residential floor plan as JSON (TarkeebAI plan_schema). '
        'Plot: 10.0 x 10.0 m. Program: 2 bedrooms, 1 bathroom, 1 kitchen, 1 living. '
        'Required adjacencies: R1-R2; R2-R3.')
print(generate(demo)[:1200])